In [3]:
import pandas as pd
import os
import sqlite3
from pathlib import Path
import shutil

#### Manually checking the code executions.

- importing all files ratio, cagr,cashflow_kpis, building_ratios, ratio_edge_case

In [4]:
import importlib
import ratio as ratio
import cagr as cagr
import cashflow_kpis as cashflow
import building_ratios as blding_ratio
import ratio_edge_case as rec

In [66]:
import importlib
import cagr           
import ratio
import building_ratios

importlib.reload(cagr)
importlib.reload(ratio)
importlib.reload(building_ratios)

from ratio import *    

### Import all required Datasets.

In [6]:
path = Path(r"C:\Users\bhrra\Desktop\nifty100\data\processed")
for file in path.glob("*.csv"):
    globals()[file.stem] = pd.read_csv(file)
a  =[]
for file in path.iterdir():
	a.append(file.name[:-4])

In [7]:
cashflow

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0
...,...,...,...,...,...,...,...
1180,1254,ZYDUSLIFE,Mar 2020,2932.0,-1004.0,-1528.0,399.0
1181,1255,ZYDUSLIFE,Mar 2021,3294.0,-837.0,-2548.0,-90.0
1182,1256,ZYDUSLIFE,Mar 2022,2104.0,-1018.0,-868.0,219.0
1183,1257,ZYDUSLIFE,Mar 2023,2689.0,1542.0,-4400.0,-170.0


In [8]:
# All dfs are 
print("="*15," All table are imported ","="*15)
for i in a:
	print(i)


===============  All table are imported  ===============
analysis
balancesheet
cashflow
companies
documents
financial_ratios
market_cap
peer_groups
profitandloss
prosandcons
sectors
stock_prices


#### Day 08 — Profitability Ratios

• Write src/analytics/ratios.py — implement Net Profit Margin: net_profit / sales x 100 — return None if sales = 0

• Implement Operating Profit Margin — compute and cross-check against opm_percentage field — log if difference > 1%

• Implement Return on Equity (ROE): net_profit / (equity_capital + reserves) x 100 — return None if equityreserves <= 0

• Implement Return on Capital Employed (ROCE): EBIT / (equity + reserves + borrowings) x 100

• For companies in Financials broad_sector — use sector-relative ROCE benchmark instead of absolute threshold

• Implement Return on Assets (ROA): net_profit / total_assets x 100 — return None if total_assets = 0

• Write 8 unit tests covering: normal case, zero denominator (None), negative equity (None), OPM cross-check

mismatch

In [9]:
print(list(profitandloss.columns))
print(profitandloss.shape)
print(balancesheet.shape)

['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout']
(1156, 15)
(1312, 13)


Net profit margin

In [10]:
npm = ratio.net_profit_margin(
    profitandloss["net_profit"],
    profitandloss["sales"]
)
net_profit_margin_result = pd.DataFrame(
	{
		"Company" : profitandloss["company_id"],
		"year" : profitandloss["year"],
		"net_profit_margin_yearly" : npm
	}
)
print(net_profit_margin_result)

     Company      year  net_profit_margin_yearly
0        ABB  Dec 2012                      8.77
1        ABB  Mar 2014                      8.70
2        ABB  Mar 2015                     10.00
3        ABB  Mar 2016                      9.76
4        ABB  Mar 2017                      9.54
...      ...       ...                       ...
1151  INDIGO  Mar 2020                     -0.69
1152  INDIGO  Mar 2021                    -42.15
1153  INDIGO  Mar 2022                     -1.22
1154  INDIGO  Mar 2023                     -0.58
1155  INDIGO  Mar 2024                     11.85

[1156 rows x 3 columns]


####  opm_percentage 

In [11]:
profitandloss["computed_opm"] = operating_profit_margin(
    profitandloss["operating_profit"],
    profitandloss["sales"],
    profitandloss["opm_percentage"]
)


In [12]:
profitandloss[["company_id", "year", "computed_opm","opm_percentage"]]

,company_id,year,computed_opm,opm_percentage
0,ABB,Dec 2012,12.22,12.0
1,ABB,Mar 2014,11.73,12.0
2,ABB,Mar 2015,13.63,14.0
3,ABB,Mar 2016,13.96,14.0
4,ABB,Mar 2017,13.71,14.0
...,...,...,...,...
1151,INDIGO,Mar 2020,88.62,4069.0
1152,INDIGO,Mar 2021,99.98,2.0
1153,INDIGO,Mar 2022,97.85,558.0
1154,INDIGO,Mar 2023,88.02,6521.0


In [13]:
len(profitandloss)

1156

In [14]:
diff = set(profitandloss["computed_opm"]).intersection(profitandloss["opm_percentage"])
print(
    profitandloss.loc[
        profitandloss["opm_percentage"].isin(profitandloss["computed_opm"]),
        ["company_id", "year", "opm_percentage", "computed_opm"]
    ]
)
print(len(set(profitandloss["computed_opm"]).symmetric_difference(profitandloss["opm_percentage"])))

      company_id      year  opm_percentage  computed_opm
9            ABB  Mar 2022            22.0         22.02
11           ABB  Mar 2024            25.0         24.84
13    ADANIENSOL  Mar 2016            88.0         87.83
21    ADANIENSOL  Mar 2024            30.0         34.39
67    ADANIPOWER  Mar 2014            30.0         29.96
...          ...       ...             ...           ...
1114       WIPRO  Mar 2014            22.0         22.19
1115       WIPRO  Mar 2015            22.0         21.94
1135   ZYDUSLIFE  Mar 2016            25.0         24.74
1141   ZYDUSLIFE  Mar 2022            22.0         22.12
1142   ZYDUSLIFE  Mar 2023            22.0         22.39

[167 rows x 4 columns]
1349


In [15]:
# return on equity

profitandloss["return_on_equity"]= return_on_equity(profitandloss["net_profit"],balancesheet["equity_capital"],balancesheet["reserves"])

# return on assests

profitandloss["return_on_assests"] = return_on_assets(profitandloss["net_profit"],balancesheet["total_assets"])

# return on capital employed

profitandloss["return_on_capital_employed"] = return_on_capital_employed(
							profitandloss["operating_profit"],
							profitandloss["depreciation"],
							balancesheet["equity_capital"],
							balancesheet["reserves"],
							balancesheet["borrowings"]
					)


In [16]:
# Operating Profit Margin. remains
cashflow

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0
...,...,...,...,...,...,...,...
1180,1254,ZYDUSLIFE,Mar 2020,2932.0,-1004.0,-1528.0,399.0
1181,1255,ZYDUSLIFE,Mar 2021,3294.0,-837.0,-2548.0,-90.0
1182,1256,ZYDUSLIFE,Mar 2022,2104.0,-1018.0,-868.0,219.0
1183,1257,ZYDUSLIFE,Mar 2023,2689.0,1542.0,-4400.0,-170.0


Day 09 — Leverage & Efficiency Ratios

• Implement Debt-to-Equity: borrowings / (equity_capital + reserves) — return 0 (not None) if borrowings = 0

• Add D/E flag: if D/E > 5 and company is NOT in Financials sector — add high_leverage_flag = True

• Implement Interest Coverage Ratio: (operating_profit + other_income) / interest — return None if interest = 0

• For ICR = None (debt-free companies) — store display label Debt Free in a separate icr_label column

• Add ICR warning flag: if ICR < 1.5 — company is at risk of not covering interest payments

• Implement Net Debt: borrowings - investments (use investments as liquid asset proxy)

• Implement Asset Turnover: sales / total_assets — return None if total_assets = 0

• Write 8 unit tests: D/E debt-free returns 0, ICR interest=0 returns None, ICR label = Debt Free, high D/E flag

Day 10 — CAGR Engine — All Growth Metrics

• Write src/analytics/cagr.py — implement CAGR formula: ((end/start)^(1/n) - 1) x 100

• Compute Revenue CAGR for 3-year, 5-year,

In [17]:
cashflow

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0
...,...,...,...,...,...,...,...
1180,1254,ZYDUSLIFE,Mar 2020,2932.0,-1004.0,-1528.0,399.0
1181,1255,ZYDUSLIFE,Mar 2021,3294.0,-837.0,-2548.0,-90.0
1182,1256,ZYDUSLIFE,Mar 2022,2104.0,-1018.0,-868.0,219.0
1183,1257,ZYDUSLIFE,Mar 2023,2689.0,1542.0,-4400.0,-170.0


In [18]:
# debt_to_equity
balancesheet["debt_to_eqty"] = debt_to_equity(balancesheet["borrowings"],balancesheet["equity_capital"],balancesheet["reserves"])


# net_debt
balancesheet["net_debt"]=net_debt(balancesheet["borrowings"],balancesheet["investments"])

# assest_turnover
profitandloss["assest_turnover"] = asset_turnover(profitandloss["sales"],balancesheet["total_assets"])

#interest_coverage_ratio

profitandloss["icr"],profitandloss["icr_label"],profitandloss['at_risk_flage'] = interest_coverage_ratio(profitandloss["operating_profit"],profitandloss["other_income"],profitandloss["interest"])


In [19]:
cashflow

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0
...,...,...,...,...,...,...,...
1180,1254,ZYDUSLIFE,Mar 2020,2932.0,-1004.0,-1528.0,399.0
1181,1255,ZYDUSLIFE,Mar 2021,3294.0,-837.0,-2548.0,-90.0
1182,1256,ZYDUSLIFE,Mar 2022,2104.0,-1018.0,-868.0,219.0
1183,1257,ZYDUSLIFE,Mar 2023,2689.0,1542.0,-4400.0,-170.0


Day 10 — CAGR Engine — All Growth Metrics

• Write src/analytics/cagr.py — implement CAGR formula: ((end/start)^(1/n) - 1) x 100

• Compute Revenue CAGR for 3-year, 5-year, and 10-year windows

• Compute PAT (net profit) CAGR for 3-year, 5-year, and 10-year windows

• Compute EPS CAGR for 3-year, 5-year, and 10-year windows

• Handle all 6 CAGR edge cases — Positive+Positive: compute normally — Positive+Negative: return None with

flag DECLINE_TO_LOSS — Negative+Positive: return None with flag TURNAROUND — Negative+Negative:

return None with flag BOTH_NEGATIVE — Zero base: return None with flag ZERO_BASE — Less than n years of

data: return None with flag INSUFFICIENT

• Store CAGR flag in a separate column alongside the CAGR value (e.g. revenue_cagr_5yr_flag)

• Write 10 unit tests: normal CAGR, turnaround flag, decline-to-loss, both negative, zero base, insufficient data

In [20]:

import cagr           
importlib.reload(cagr)
from cagr import *           


In [21]:
cagr_rows = []

for company_id in companies["id"]:
    company_pl = profitandloss[profitandloss["company_id"] == company_id].sort_values("year")

    sales_series = dict(zip(company_pl["year"], company_pl["sales"]))
    pat_series = dict(zip(company_pl["year"], company_pl["net_profit"]))
    eps_series = dict(zip(company_pl["year"], company_pl["eps"]))

    row = {"company_id": company_id}

    for n in (3, 5, 10):
        row[f"revenue_cagr_{n}yr"], row[f"revenue_cagr_{n}yr_flag"] = revenue_cagr(sales_series, n)
        row[f"pat_cagr_{n}yr"], row[f"pat_cagr_{n}yr_flag"] = pat_cagr(pat_series, n)
        row[f"eps_cagr_{n}yr"], row[f"eps_cagr_{n}yr_flag"] = eps_cagr(eps_series, n)

    cagr_rows.append(row)

cagr_df = pd.DataFrame(cagr_rows)

In [22]:
cashflow

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0
...,...,...,...,...,...,...,...
1180,1254,ZYDUSLIFE,Mar 2020,2932.0,-1004.0,-1528.0,399.0
1181,1255,ZYDUSLIFE,Mar 2021,3294.0,-837.0,-2548.0,-90.0
1182,1256,ZYDUSLIFE,Mar 2022,2104.0,-1018.0,-868.0,219.0
1183,1257,ZYDUSLIFE,Mar 2023,2689.0,1542.0,-4400.0,-170.0


In [23]:
cagr_df

,company_id,revenue_cagr_3yr,revenue_cagr_3yr_flag,pat_cagr_3yr,pat_cagr_3yr_flag,eps_cagr_3yr,eps_cagr_3yr_flag,revenue_cagr_5yr,revenue_cagr_5yr_flag,pat_cagr_5yr,pat_cagr_5yr_flag,eps_cagr_5yr,eps_cagr_5yr_flag,revenue_cagr_10yr,revenue_cagr_10yr_flag,pat_cagr_10yr,pat_cagr_10yr_flag,eps_cagr_10yr,eps_cagr_10yr_flag
0,ABB,10.71,None,20.23,None,20.24,None,9.72,None,21.69,None,21.66,None,9.90,None,19.75,None,19.77,None
1,ADANIENSOL,18.72,None,-2.49,None,-3.13,None,17.85,None,16.43,None,14.87,None,NaN,ZERO_BASE,NaN,ZERO_BASE,NaN,ZERO_BASE
2,ADANIENT,34.60,None,47.18,None,51.83,None,19.02,None,45.81,None,31.95,None,5.78,None,2.34,None,3.42,None
3,ADANIGREEN,43.44,None,90.59,None,91.29,None,34.98,None,NaN,TURNAROUND,NaN,TURNAROUND,NaN,INSUFFICIENT,NaN,INSUFFICIENT,NaN,INSUFFICIENT
4,ADANIPORTS,28.63,None,17.08,None,14.98,None,19.58,None,14.91,None,14.87,None,18.65,None,16.62,None,16.86,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,TCS,13.63,None,12.29,None,13.01,None,10.46,None,7.87,None,8.62,None,11.40,None,9.08,None,9.99,None
85,TECHM,11.16,None,-18.04,None,-19.50,None,8.40,None,-10.99,None,-11.42,None,10.71,None,-2.42,None,-2.84,None
86,TITAN,33.14,None,53.11,None,52.48,None,20.90,None,20.27,None,19.51,None,16.68,None,16.88,None,17.16,None
87,TORNTPHARM,10.25,None,9.77,None,9.82,None,6.93,None,30.59,None,30.39,None,9.90,None,9.57,None,9.37,None


In [24]:
import cagr           
importlib.reload(cagr)
from cashflow_kpis import *

In [25]:
company_cf = cashflow[
    cashflow["company_id"] == company_id
].sort_values("year")

company_pl = profitandloss[
    profitandloss["company_id"] == company_id
].sort_values("year")

for _, cf_row in company_cf.iterrows():
    year = cf_row["year"]

    fcf = free_cash_flow(
        cf_row["operating_activity"],
        cf_row["investing_activity"]
    )

    sales_row = company_pl[company_pl["year"] == year]

    sales = sales_row["sales"].iloc[0] if not sales_row.empty else 0

    capex_pct, capex_label = capex_intensity(
        cf_row["investing_activity"],
        sales
    )

In [26]:
# company_cf = cashflow[cashflow["company_id"] == company_id].sort_values("year")
# company_pl = profitandloss[profitandloss["company_id"] == company_id].sort_values("year")

# for _, cashflow in company_cf.iterrows():
#     year = cashflow["year"]
#     fcf = free_cash_flow(cashflow["operating_activity"], cashflow["investing_activity"])

#     # capex_intensity ko sales chahiye -- same year ka profitandloss se lo
#     sales_row = company_pl[company_pl["year"] == year]
#     sales = sales_row["sales"].iloc[0] if not sales_row.empty else 0
#     capex_pct, capex_label = capex_intensity(cashflow["investing_activity"], sales)

In [27]:
cashflow

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0
...,...,...,...,...,...,...,...
1180,1254,ZYDUSLIFE,Mar 2020,2932.0,-1004.0,-1528.0,399.0
1181,1255,ZYDUSLIFE,Mar 2021,3294.0,-837.0,-2548.0,-90.0
1182,1256,ZYDUSLIFE,Mar 2022,2104.0,-1018.0,-868.0,219.0
1183,1257,ZYDUSLIFE,Mar 2023,2689.0,1542.0,-4400.0,-170.0


In [28]:
last_5_cf = company_cf.tail(5)
last_5_pl = company_pl.tail(5)

cfo_5yr_avg = last_5_cf["operating_activity"].mean()
pat_5yr_avg = last_5_pl["net_profit"].mean()

quality_ratio, quality_label = cfo_quality_score(cfo_5yr_avg, pat_5yr_avg)

In [29]:
latest_cf = company_cf.iloc[-1]  # sabse recent year

cfo_sign, cfi_sign, cff_sign, pattern_label = classify_capital_allocation(
    latest_cf["operating_activity"],
    latest_cf["investing_activity"],
    latest_cf["financing_activity"],
    cfo_quality_ratio=quality_ratio  # Step 2 se
)

In [30]:
profitandloss

,id,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,eps,dividend_payout,computed_opm,return_on_equity,return_on_assests,return_on_capital_employed,assest_turnover,icr,icr_label,at_risk_flage
0,61,ABB,Dec 2012,1653,1451,202.0,12.0,33,0,19,...,68.0,25.0,12.22,22.41,15.99,28.28,1.82,NaN,Debt Free,False
1,62,ABB,Mar 2014,2276,2009,267.0,12.0,49,0,22,...,93.0,25.0,11.73,25.13,17.38,31.09,2.00,NaN,Debt Free,False
2,63,ABB,Mar 2015,2289,1977,312.0,14.0,48,0,15,...,108.0,29.0,13.63,24.44,16.67,31.70,1.67,NaN,Debt Free,False
3,64,ABB,Mar 2016,2614,2250,365.0,14.0,50,3,14,...,120.0,29.0,13.96,21.34,15.78,29.37,1.62,138.33,None,False
4,65,ABB,Mar 2017,2903,2505,398.0,14.0,57,2,16,...,130.0,31.0,13.71,19.97,13.41,27.54,1.41,227.50,None,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1151,1369,INDIGO,Mar 2020,35756,1902,31687.0,4069.0,11,1531,3974,...,-6.0,0.0,88.62,-0.53,-0.37,55.06,0.53,20.70,None,False
1152,1370,INDIGO,Mar 2021,14641,2170,14638.0,2.0,0,1036,4699,...,-151.0,0.0,99.98,-12.85,-8.84,20.16,0.21,14.13,None,False
1153,1371,INDIGO,Mar 2022,25931,2386,25373.0,558.0,2,725,5068,...,-160.0,0.0,97.85,-0.57,-0.39,32.29,0.32,35.00,None,False
1154,1372,INDIGO,Mar 2023,54446,3168,47926.0,6521.0,12,1431,5101,...,-8.0,0.0,88.02,-0.50,-0.37,63.97,0.64,33.50,None,False


In [31]:
print(type(cashflow))
print(type(profitandloss))
print(type(companies))

print(cashflow.columns if hasattr(cashflow, "columns") else "No columns")
print(profitandloss.columns if hasattr(profitandloss, "columns") else "No columns")

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
Index(['id', 'company_id', 'year', 'operating_activity', 'investing_activity',
       'financing_activity', 'net_cash_flow'],
      dtype='object')
Index(['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit',
       'opm_percentage', 'other_income', 'interest', 'depreciation',
       'profit_before_tax', 'tax_percentage', 'net_profit', 'eps',
       'dividend_payout', 'computed_opm', 'return_on_equity',
       'return_on_assests', 'return_on_capital_employed', 'assest_turnover',
       'icr', 'icr_label', 'at_risk_flage'],
      dtype='object')


In [65]:
capital_allocation_rows = []

for company_id in companies["id"]:
    company_cf = cashflow[cashflow["company_id"] == company_id].sort_values("year")
    company_pl = profitandloss[profitandloss["company_id"] == company_id].sort_values("year")

    if company_cf.empty:
        continue

    last_5_cf = company_cf.tail(5)
    last_5_pl = company_pl.tail(5)
    cfo_5yr_avg = last_5_cf["operating_activity"].mean()
    pat_5yr_avg = last_5_pl["net_profit"].mean()
    quality_ratio, quality_label = cfo_quality_score(cfo_5yr_avg, pat_5yr_avg)

    latest_cf = company_cf.iloc[-1]
    cfo_sign, cfi_sign, cff_sign, pattern_label = classify_capital_allocation(
        latest_cf["operating_activity"], latest_cf["investing_activity"],
        latest_cf["financing_activity"], cfo_quality_ratio=quality_ratio
    )

    capital_allocation_rows.append({
        "company_id": company_id,
        "year": latest_cf["year"],
        "cfo_sign": cfo_sign,
        "cfi_sign": cfi_sign,
        "cff_sign": cff_sign,
        "pattern_label": pattern_label,
    })

capital_allocation_df = pd.DataFrame(capital_allocation_rows)

store_path = Path(r"C:\Users\bhrra\Desktop\nifty100\output")
capital_allocation_df.to_csv(store_path/"capital_allocation.csv", index=False)
capital_allocation_df

,company_id,year,cfo_sign,cfi_sign,cff_sign,pattern_label
0,ABB,Mar 2024,1,-1,-1,Shareholder Returns
1,ADANIENSOL,Mar 2024,1,-1,-1,Shareholder Returns
2,ADANIENT,Mar 2024,1,-1,1,Mixed
3,ADANIGREEN,Mar 2024,1,-1,1,Mixed
4,ADANIPORTS,Mar 2024,1,-1,-1,Shareholder Returns
...,...,...,...,...,...,...
83,TCS,Mar-24,1,1,-1,Liquidating Assets
84,TECHM,Mar 2024,1,-1,-1,Shareholder Returns
85,TITAN,Mar 2024,1,-1,-1,Reinvestor
86,TORNTPHARM,Mar 2024,1,-1,-1,Shareholder Returns


Day 12 — Populate financial_ratios Table

• Run the full ratio engine for all 92 companies across all available years

• Write all 14+ computed KPI columns into the financial_ratios table in SQLite

• KPI columns required: net_profit_margin_pct, operating_profit_margin_pct, return_on_equity_pct,

debt_to_equity, interest_coverage, asset_turnover, free_cash_flow_cr, capex_cr, earnings_per_share,

book_value_per_share, dividend_payout_ratio_pct, total_debt_cr, cash_from_operations_cr, revenue_cagr_5yr,

pat_cagr_5yr, eps_cagr_5yr, composite_quality_score

• Verify row count: SELECT COUNT(*) FROM financial_ratios — must be >= 1,100 rows

• Manual spot-check: for 3 companies, recompute ROE and 5-year Revenue CAGR manually in a spreadsheet and

compare to database values — difference must be less than 0.1%

In [33]:
cashflow

,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
0,37,TCS,Mar-13,11615.0,-6038.0,-5729.0,-152.0
1,38,TCS,Mar-14,14751.0,-9452.0,-5673.0,-374.0
2,39,TCS,Mar-15,19369.0,-1807.0,-17168.0,394.0
3,40,TCS,Mar-16,19109.0,-5010.0,-9666.0,4433.0
4,41,TCS,Mar-17,25223.0,-16895.0,-11026.0,-2698.0
...,...,...,...,...,...,...,...
1180,1254,ZYDUSLIFE,Mar 2020,2932.0,-1004.0,-1528.0,399.0
1181,1255,ZYDUSLIFE,Mar 2021,3294.0,-837.0,-2548.0,-90.0
1182,1256,ZYDUSLIFE,Mar 2022,2104.0,-1018.0,-868.0,219.0
1183,1257,ZYDUSLIFE,Mar 2023,2689.0,1542.0,-4400.0,-170.0


In [34]:
dupe_check = cashflow[cashflow.duplicated(subset=["company_id", "year"], keep=False)]
print(len(dupe_check))
dupe_check.head(10)

46


,id,company_id,year,operating_activity,investing_activity,financing_activity,net_cash_flow
13,62,ABB,Mar 2014,155.0,-144.0,-42.0,-31.0
14,63,ABB,Mar 2015,215.0,-187.0,-58.0,-30.0
15,64,ABB,Mar 2016,249.0,-77.0,-80.0,91.0
16,65,ABB,Mar 2017,307.0,-155.0,-90.0,62.0
17,66,ABB,Mar 2018,153.0,-215.0,-102.0,-165.0
18,67,ABB,Mar 2019,499.0,-257.0,-143.0,99.0
19,68,ABB,Mar 2020,626.0,-401.0,-217.0,8.0
20,69,ABB,Mar 2021,727.0,-72.0,-582.0,73.0
21,70,ABB,Mar 2022,948.0,-396.0,-637.0,-86.0
22,71,ABB,Mar 2023,893.0,-148.0,-639.0,107.0


In [35]:
print(len(cashflow))
cashflow = cashflow.drop_duplicates(subset=["company_id", "year"], keep="last")
print(len(cashflow))

1185
1162


In [36]:
# Profit & Loss
pl_before = len(profitandloss)
profitandloss = profitandloss.drop_duplicates(
    subset=["company_id", "year"],
    keep="last"
)
pl_after = len(profitandloss)

print(f"Profit & Loss: {pl_before} → {pl_after} rows "
      f"({pl_before - pl_after} duplicates removed)")


# Balance Sheet
bs_before = len(balancesheet)
balancesheet = balancesheet.drop_duplicates(
    subset=["company_id", "year"],
    keep="last"
)
bs_after = len(balancesheet)

print(f"Balance Sheet: {bs_before} → {bs_after} rows "
      f"({bs_before - bs_after} duplicates removed)")


# Cash Flow
cf_before = len(cashflow)
cashflow = cashflow.drop_duplicates(
    subset=["company_id", "year"],
    keep="last"
)
cf_after = len(cashflow)

print(f"Cash Flow: {cf_before} → {cf_after} rows "
      f"({cf_before - cf_after} duplicates removed)")

Profit & Loss: 1156 → 1144 rows (12 duplicates removed)
Balance Sheet: 1312 → 1225 rows (87 duplicates removed)
Cash Flow: 1162 → 1162 rows (0 duplicates removed)


In [39]:
financial_ratio_rows = []

financials_ids = set(
    sectors[sectors["broad_sector"] == "Financials"]["company_id"]
)

for company_id in companies["id"]:

    company_pl = (
        profitandloss[profitandloss["company_id"] == company_id]
        .sort_values("year")
    )

    company_bs = (
        balancesheet[balancesheet["company_id"] == company_id]
        .set_index("year")
    )

    company_cf = (
        cashflow[cashflow["company_id"] == company_id]
        .set_index("year")
    )

    if company_pl.empty:
        continue

    # CAGR -- company-level, calculate once
    sales_series = dict(zip(company_pl["year"], company_pl["sales"]))
    pat_series = dict(zip(company_pl["year"], company_pl["net_profit"]))
    eps_series = dict(zip(company_pl["year"], company_pl["eps"]))

    rev_cagr_5, rev_flag_5 = revenue_cagr(sales_series, 5)
    pat_cagr_5, pat_flag_5 = pat_cagr(pat_series, 5)
    eps_cagr_5, eps_flag_5 = eps_cagr(eps_series, 5)

    is_fin = company_id in financials_ids
    latest_year = company_pl["year"].max()

    # IMPORTANT: use pl_row, not profitandloss
    for _, pl_row in company_pl.iterrows():

        year = pl_row["year"]

        if year not in company_bs.index or year not in company_cf.index:
            continue

        # IMPORTANT: use bs_row and cf_row
        bs_row = company_bs.loc[year]
        cf_row = company_cf.loc[year]

        # Day 08 -- profitability
        npm = net_profit_margin(
            pl_row["net_profit"],
            pl_row["sales"]
        )

        opm = operating_profit_margin(
            pl_row["operating_profit"],
            pl_row["sales"]
        )

        roe = return_on_equity(
            pl_row["net_profit"],
            bs_row["equity_capital"],
            bs_row["reserves"]
        )

        # Day 09 -- leverage / efficiency
        de = debt_to_equity(
            bs_row["borrowings"],
            bs_row["equity_capital"],
            bs_row["reserves"],
            is_financials=is_fin
        )

        icr, icr_label, icr_risk = interest_coverage_ratio(
            pl_row["operating_profit"],
            pl_row.get("other_income", 0),
            bs_row.get("interest", 0)
        )

        at = asset_turnover(
            pl_row["sales"],
            bs_row["total_assets"]
        )

        # Day 11 -- cash flow
        fcf = free_cash_flow(
            cf_row["operating_activity"],
            cf_row["investing_activity"]
        )

        capex_pct, capex_label = capex_intensity(
            cf_row["investing_activity"],
            pl_row["sales"]
        )

        cfo_sign, cfi_sign, cff_sign, pattern_label = (
            classify_capital_allocation(
                cf_row["operating_activity"],
                cf_row["investing_activity"],
                cf_row["financing_activity"]
            )
        )

        row = {
            "company_id": company_id,
            "year": year,

            "net_profit_margin_pct": npm,
            "operating_profit_margin_pct": opm,
            "return_on_equity_pct": roe,

            "debt_to_equity": de,
            

            "interest_coverage": icr,
            "icr_label": icr_label,

            "asset_turnover": at,

            "free_cash_flow_cr": fcf,
            "capex_cr": abs(cf_row["investing_activity"]),
            "capex_intensity_pct": capex_pct,

            "earnings_per_share": pl_row.get("eps"),
            "dividend_payout_ratio_pct": pl_row.get("dividend_payout"),

            "total_debt_cr": bs_row["borrowings"],
            "cash_from_operations_cr": cf_row["operating_activity"],

            "capital_allocation_label": pattern_label,

            # CAGR -- only latest year
            "revenue_cagr_5yr": (
                rev_cagr_5 if year == latest_year else None
            ),

            "pat_cagr_5yr": (
                pat_cagr_5 if year == latest_year else None
            ),

            "eps_cagr_5yr": (
                eps_cagr_5 if year == latest_year else None
            ),
        }

        financial_ratio_rows.append(row)


financial_ratios_df = pd.DataFrame(financial_ratio_rows)

print(f"Total rows built: {len(financial_ratios_df)}")


BASE_DIR = Path(r"C:\Users\bhrra\Desktop\nifty100")
DATA_DIR = BASE_DIR / "data" / "processed"
DB_PATH = BASE_DIR / "db" / "nifty100.db"

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")


# ============================================================
# Add missing columns to financial_ratios table
# ============================================================

# Get existing columns from SQLite
db_columns = pd.read_sql_query(
    "PRAGMA table_info(financial_ratios)",
    conn
)["name"].tolist()

print("Existing DB columns:")
print(db_columns)


# Define datatype for new columns
column_types = {
    "icr_label": "TEXT",
    "capex_intensity_pct": "REAL",
    "capital_allocation_label": "TEXT",
    "revenue_cagr_5yr": "REAL",
    "pat_cagr_5yr": "REAL",
    "eps_cagr_5yr": "REAL"
}


# Add only columns that don't already exist
for column, dtype in column_types.items():

    if column not in db_columns:

        conn.execute(
            f'ALTER TABLE financial_ratios '
            f'ADD COLUMN "{column}" {dtype}'
        )

        print(f"Added column: {column}")


conn.commit()


# ============================================================
# Check final schema
# ============================================================

print("\nUpdated financial_ratios schema:")

schema_df = pd.read_sql_query(
    "PRAGMA table_info(financial_ratios)",
    conn
)

print(schema_df[["name", "type"]])


# ============================================================
# Check DataFrame columns
# ============================================================

print("\nDataFrame columns:")
print(financial_ratios_df.columns.tolist())


# ============================================================
# Insert data
# ============================================================

financial_ratios_df.to_sql(
    "financial_ratios",
    conn,
    if_exists="append",
    index=False
)

conn.commit()


# ============================================================
# Verify row count
# ============================================================

count = conn.execute(
    "SELECT COUNT(*) FROM financial_ratios"
).fetchone()[0]

print(f"\nfinancial_ratios row count: {count}")

conn.close()

Total rows built: 983
Existing DB columns:
['id', 'company_id', 'year', 'net_profit_margin_pct', 'operating_profit_margin_pct', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'free_cash_flow_cr', 'capex_cr', 'earnings_per_share', 'book_value_per_share', 'dividend_payout_ratio_pct', 'total_debt_cr', 'cash_from_operations_cr']
Added column: icr_label
Added column: capex_intensity_pct
Added column: capital_allocation_label
Added column: revenue_cagr_5yr
Added column: pat_cagr_5yr
Added column: eps_cagr_5yr

Updated financial_ratios schema:
                           name     type
0                            id  INTEGER
1                    company_id     TEXT
2                          year     TEXT
3         net_profit_margin_pct     REAL
4   operating_profit_margin_pct     REAL
5          return_on_equity_pct     REAL
6                debt_to_equity     REAL
7             interest_coverage     REAL
8                asset_turnover     REAL
9            

In [40]:
profitandloss

,id,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,eps,dividend_payout,computed_opm,return_on_equity,return_on_assests,return_on_capital_employed,assest_turnover,icr,icr_label,at_risk_flage
0,61,ABB,Dec 2012,1653,1451,202.0,12.0,33,0,19,...,68.0,25.0,12.22,22.41,15.99,28.28,1.82,NaN,Debt Free,False
1,62,ABB,Mar 2014,2276,2009,267.0,12.0,49,0,22,...,93.0,25.0,11.73,25.13,17.38,31.09,2.00,NaN,Debt Free,False
2,63,ABB,Mar 2015,2289,1977,312.0,14.0,48,0,15,...,108.0,29.0,13.63,24.44,16.67,31.70,1.67,NaN,Debt Free,False
3,64,ABB,Mar 2016,2614,2250,365.0,14.0,50,3,14,...,120.0,29.0,13.96,21.34,15.78,29.37,1.62,138.33,None,False
4,65,ABB,Mar 2017,2903,2505,398.0,14.0,57,2,16,...,130.0,31.0,13.71,19.97,13.41,27.54,1.41,227.50,None,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1151,1369,INDIGO,Mar 2020,35756,1902,31687.0,4069.0,11,1531,3974,...,-6.0,0.0,88.62,-0.53,-0.37,55.06,0.53,20.70,None,False
1152,1370,INDIGO,Mar 2021,14641,2170,14638.0,2.0,0,1036,4699,...,-151.0,0.0,99.98,-12.85,-8.84,20.16,0.21,14.13,None,False
1153,1371,INDIGO,Mar 2022,25931,2386,25373.0,558.0,2,725,5068,...,-160.0,0.0,97.85,-0.57,-0.39,32.29,0.32,35.00,None,False
1154,1372,INDIGO,Mar 2023,54446,3168,47926.0,6521.0,12,1431,5101,...,-8.0,0.0,88.02,-0.50,-0.37,63.97,0.64,33.50,None,False


In [41]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")

print("Using database:")
print(DB_PATH)

print("Exists:", DB_PATH.exists())

conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", conn)

print("\nTables:")
print(tables)

for table in tables["name"]:
    count = conn.execute(
        f'SELECT COUNT(*) FROM "{table}"'
    ).fetchone()[0]

    print(f"{table:30} → {count:,} rows")

conn.close()

Using database:
C:\Users\bhrra\Desktop\nifty100\db\nifty100.db
Exists: True

Tables:
                name
0           analysis
1       balancesheet
2           cashflow
3          companies
4          documents
5   financial_ratios
6         market_cap
7        peer_groups
8      profitandloss
9        prosandcons
10           sectors
11      stock_prices
analysis                       → 60 rows
balancesheet                   → 3,762 rows
cashflow                       → 3,509 rows
companies                      → 267 rows
documents                      → 4,599 rows
financial_ratios               → 4,100 rows
market_cap                     → 1,656 rows
peer_groups                    → 168 rows
profitandloss                  → 3,444 rows
prosandcons                    → 30 rows
sectors                        → 276 rows
stock_prices                   → 16,560 rows


Day 13 — Bank ROCE Carve-Out & Edge Case Log

• For the 19 companies in the Financials broad_sector (banks, NBFCs, insurance) — standard D/E warning flag is

suppressed because high leverage is structurally normal

• Cross-check computed ROCE against the pre-computed roce_percentage column in companies.xlsx — log

anomalies where difference > 5% to output/ratio_edge_cases.log

• Cross-check computed ROE against the pre-computed roe_percentage column in companies.xlsx — note that

some source values appear anomalous (e.g. TCS shows 0.52) — use ratio engine value for analytics, source value

for display only

• Review ratio_edge_cases.log and categorise each anomaly as: data source issue, version difference, or formula

discrepancy

In [42]:
import sqlite3
from pathlib import Path


conn = sqlite3.connect(DB_PATH)

# Get all table names
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

print("Tables in database:")
print(tables)

print("\n" + "=" * 60)

# Check each table
for table in tables["name"]:
    
    # Row count
    count = conn.execute(
        f'SELECT COUNT(*) FROM "{table}"'
    ).fetchone()[0]

    # Columns
    columns = pd.read_sql_query(
        f'PRAGMA table_info("{table}")',
        conn
    )

    print(f"\n📌 Table: {table}")
    print(f"Rows: {count}")
    print(f"Columns: {len(columns)}")
    print("Column names:")
    print(columns["name"].tolist())

# conn.close()

Tables in database:
                name
0           analysis
1       balancesheet
2           cashflow
3          companies
4          documents
5   financial_ratios
6         market_cap
7        peer_groups
8      profitandloss
9        prosandcons
10           sectors
11      stock_prices


📌 Table: analysis
Rows: 60
Columns: 6
Column names:
['id', 'company_id', 'compounded_sales_growth', 'compounded_profit_growth', 'stock_price_cagr', 'roe']

📌 Table: balancesheet
Rows: 3762
Columns: 13
Column names:
['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip', 'investments', 'other_asset', 'total_assets']

📌 Table: cashflow
Rows: 3509
Columns: 7
Column names:
['id', 'company_id', 'year', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow']

📌 Table: companies
Rows: 267
Columns: 12
Column names:
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', '

In [59]:
import ratio_edge_case
importlib.reload(ratio_edge_case)
from ratio_edge_case import run_edge_case_audit

In [61]:
fr_ids = set(
    fr_check["company_id"].astype(str).str.strip()
)

comp_ids = set(
    comp_check["company_id"].astype(str).str.strip()
)

missing_ids = fr_ids - comp_ids

print("Financial ratio company IDs:", len(fr_ids))
print("Companies IDs:", len(comp_ids))
print("Missing IDs:", len(missing_ids))
print(sorted(missing_ids))

Financial ratio company IDs: 91
Companies IDs: 89
Missing IDs: 5
['INDIGO', 'LTIM', 'TVSMOTOR', 'ULTRACEMCO', 'UNIONBANK']


In [60]:
fr_check = pd.read_sql("SELECT * FROM financial_ratios", conn)
comp_check = pd.read_sql("SELECT id, roce_percentage, roe_percentage FROM companies", conn)
comp_check = comp_check.rename(columns={"id": "company_id"})

log_path = Path(r"C:\Users\bhrra\Desktop\nifty100\output")

run_edge_case_audit(
    fr_check,
    comp_check,
    sectors,
    log_path / "ratio_edge_cases.log"
)

18 anomalies logged to C:\Users\bhrra\Desktop\nifty100\output\ratio_edge_cases.log
Financials sector companies (D/E flag suppressed): 23


Day 14 — Tests & Sprint Review

• Run all 20 KPI formula unit tests — must show 0 failures

• Review ratio_edge_cases.log — confirm all anomalies are documented and understood

• Run screener preview: quick filter ROE > 15% and D/E < 1 — verify result count is between 15 and 50 companies

and the list makes business sense

• Sprint 2 retrospective — document formula decisions and edge case resolutions for team knowledge base

• Demo financial_ratios table to team lead — show 5 companies with all computed KPIs

In [62]:
# Terminal mein (notebook nahi):
# pytest tests/kpi/test_ratio_formulas.py -v

# Screener preview -- notebook/DB Browser mein:
screener_result = pd.read_sql("""
    SELECT company_id, year, return_on_equity_pct, debt_to_equity
    FROM financial_ratios
    WHERE year = (SELECT MAX(year) FROM financial_ratios)
      AND return_on_equity_pct > 15
      AND debt_to_equity < 1
    ORDER BY return_on_equity_pct DESC
""", conn)

print(f"Screener matched: {len(screener_result)} companies")
screener_result.head(10)

conn.close()

Screener matched: 4 companies


# For checking Kpis
# -----  run pytest tests\kpi\test_ratio_formulas.py -v in cmd

In [68]:
import sqlite3

conn = sqlite3.connect(DB_PATH)

cursor = conn.execute("""
    SELECT *
    FROM financial_ratios
""")

for row in cursor:
    print(row)

conn.close()

(6, 'ABB', 'Mar 2016', 9.76, 14.0, 21.34, 0.0, 138.3333, 1.6176, 172.0, 77.0, 120.0, 5.6905, 29.0, 0, 249.0, None, None, None, None, None, None)
(7, 'ABB', 'Mar 2016', 9.76, 14.0, 21.34, 0.0, 138.3333, 1.6176, 728.0, 816.0, 120.0, 5.6905, 29.0, 0, 1544.0, None, None, None, None, None, None)
(8, 'ABB', 'Mar 2017', 9.54, 14.0, 19.97, 0.0, 227.5, 1.4051, 152.0, 155.0, 130.0, 6.6048, 31.0, 0, 307.0, None, None, None, None, None, None)
(9, 'ABB', 'Mar 2017', 9.54, 14.0, 19.97, 0.0, 227.5, 1.4051, 459.0, 1730.0, 130.0, 6.6048, 31.0, 0, 2189.0, None, None, None, None, None, None)
(10, 'ABB', 'Mar 2018', 12.16, 16.0, 23.69, 0.0, 160.5, 1.3651, -62.0, 215.0, 189.0, 8.0619, 29.0, 0, 153.0, None, None, None, None, None, None)
(11, 'ABB', 'Mar 2018', 12.16, 16.0, 23.69, 0.0, 160.5, 1.3651, -994.0, 3192.0, 189.0, 8.0619, 29.0, 0, 2198.0, None, None, None, None, None, None)
(12, 'ABB', 'Mar 2019', 12.23, 16.0, 22.41, 0.0, 359.0, 1.2509, 242.0, 257.0, 212.0, 9.5619, 31.0, 0, 499.0, None, None, None, 